## Przystosowanie geostroficzne w równaniach płytkiej wody

**Pytanie:** jaka część początkowego wzniesienia powierzchni płynu pozostaje po swobodnej ewolucji w obracającym się układzie i jak zależy to od stosunku rozmiaru zaburzenia $a$ do promienia deformacji Rossby'ego $L_d=\sqrt{gH}/f$?

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA')

In [ ]:
import os
os.environ['NUMBA_THREADING_LAYER'] = 'workqueue'

import numpy as np
from matplotlib import pyplot
from matplotlib.colors import TwoSlopeNorm
from open_atmos_jupyter_utils import show_plot, show_anim

pyplot.rcParams.update({'font.size': 14, 'legend.fontsize': 11})

from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions

### Równania

Oznaczenia jak u Vallisa: $h=H+\eta$ (wysokość kolumny płynu, $\eta$ – wychylenie powierzchni), płaskie dno, stały parametr Coriolisa $f$:

$$
\begin{cases}
  \partial_t h &=&\! -\nabla \cdot (\vec{u}h)\\
  \partial_t (hu) &=&\! -\nabla \cdot (\vec{u}hu) - gh\partial_x h + f\,hv \\
  \partial_t (hv) &=&\! -\nabla \cdot (\vec{u}hv) - gh\partial_y h - f\,hu
\end{cases}
$$

W przybliżeniu liniowym wirowość potencjalna $q=\partial_x v-\partial_y u-f\eta/H$ nie zmienia się w czasie, a stan końcowy jest w równowadze geostroficznej ($fv=g\partial_x\eta$, $fu=-g\partial_y\eta$). Stąd (Vallis, 3.164):

$$\tilde\eta_{k,l}=\frac{\tilde\eta_{k,l}(0)}{1+(k^2+l^2)L_d^2}$$

Dla $f=0$ zaburzenie odpływa całkowicie w postaci fal o prędkości $c=\sqrt{gH}$.

### Solwer

Solwer z zajęć (`ShallowWaterEquationsIntegrator`) rozszerzony o:
- parametr $g$ w członie ciśnieniowym,
- człon Coriolisa: obrót wektora $(hu, hv)$ o kąt $f\Delta t/2$ przed i po adwekcji pędu (dokładne rozwiązanie $\partial_t(hu)=f\,hv$, $\partial_t(hv)=-f\,hu$),
- periodyczne warunki brzegowe: liczby Couranta i gradient $h$ liczone okresowo także na brzegach obszaru (wymagane, by przepływ geostroficzny wzdłuż $y$ mógł przechodzić przez brzeg).

In [ ]:
class RotatingShallowWaterEquationsIntegrator:
    def __init__(self, *, h0, options=None):
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        X, Y, grid = 0, 1, h0.shape
        stepper = Stepper(options=options, grid=grid)
        boundary_cond = [boundary_conditions.Periodic()] * len(grid)
        advectees = {
            "h": ScalarField(h0, options.n_halo, boundary_cond),
            "uh": ScalarField(np.zeros(grid), options.n_halo, boundary_cond),
            "vh": ScalarField(np.zeros(grid), options.n_halo, boundary_cond),
        }
        self.advector = VectorField((
                np.zeros((grid[X] + 1, grid[Y])),
                np.zeros((grid[X], grid[Y] + 1))
            ), options.n_halo, boundary_cond
        )
        self.solvers = {k: Solver(stepper, v, self.advector) for k, v in advectees.items()}

    def __getitem__(self, key):
        return self.solvers[key].advectee.get()

    def _apply_half_rhs(self, *, key, g, dt, axis, grid_step):
        h = self['h']
        self[key][:] -= g * dt / 2 * h * (np.roll(h, -1, axis) - np.roll(h, 1, axis)) / (2 * grid_step[axis])

    def _apply_half_coriolis(self, *, f, dt):
        uh, vh = self['uh'].copy(), self['vh'].copy()
        cos, sin = np.cos(f * dt / 2), np.sin(f * dt / 2)
        self['uh'][:] = cos * uh + sin * vh
        self['vh'][:] = cos * vh - sin * uh

    def _update_courant_numbers(self, *, axis, key, mask, dt, grid_step):
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)

        velocity_at_cell_boundaries = (velocity + np.roll(velocity, 1, axis)) / 2
        courant_number = self.advector.get_component(axis)
        courant_number[:] = np.concatenate(
            (velocity_at_cell_boundaries, velocity_at_cell_boundaries.take([0], axis=axis)), axis=axis
        ) * dt / grid_step[axis]
        assert np.amax(np.abs(courant_number)) <= 1

    def __call__(self, *, nt, g, f, grid_step, dt, outfreq, eps=1e-7):
        output = {k: [] for k in self.solvers.keys()}
        for it in range(nt + 1):
            if it != 0:
                mask = self['h'] > eps
                for axis, key in enumerate(("uh", "vh")):
                    self._update_courant_numbers(axis=axis, key=key, mask=mask, dt=dt, grid_step=grid_step)
                self.solvers["h"].advance(n_steps=1)
                self._apply_half_coriolis(f=f, dt=dt)
                for axis, key in enumerate(("uh", "vh")):
                    self._apply_half_rhs(key=key, g=g, dt=dt, axis=axis, grid_step=grid_step)
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(key=key, g=g, dt=dt, axis=axis, grid_step=grid_step)
                self._apply_half_coriolis(f=f, dt=dt)
            if it % outfreq == 0:
                for key in self.solvers.keys():
                    output[key].append(self[key].copy())
        return output

### Parametry, warunek początkowy i teoria

$g=10$, $H=1$, $L_d=1$ (stąd $f=\sqrt{gH}/L_d$), wysokość garbu $\eta_0=0.02H$ (reżim liniowy). Krok siatki $\Delta x=\Delta y=L_d/N$, krok czasowy z liczby Couranta fal $c\,\Delta t/\Delta x$. Czas mierzony w okresach inercyjnych $T_f=2\pi/f$. Stan końcowy symulacji to średnia z ostatniego okresu inercyjnego (usuwa oscylacje inercyjne).

In [ ]:
g, H, Ld, eta_0 = 10., 1., 1., .02
c = np.sqrt(g * H)
f = c / Ld
T_f = 2 * np.pi / f


def tophat(r, *, a, w):
    return eta_0 * (1 - np.tanh((r - a) / w)) / 2


def channel(profile, ny=4):
    return np.repeat(profile[:, None], ny, axis=1)


def adjusted_state(eta_initial, dx):
    k = np.meshgrid(*(2 * np.pi * np.fft.fftfreq(n, d=dx) for n in eta_initial.shape), indexing='ij')
    return np.fft.ifftn(np.fft.fftn(eta_initial) / (1 + Ld**2 * sum(ki**2 for ki in k))).real


def simulate(*, eta_initial, dx, rotation=True, n_periods=4, courant=.32, outfreq=5):
    dt = courant * dx / c
    output = RotatingShallowWaterEquationsIntegrator(h0=H + eta_initial)(
        nt=int(round(n_periods * T_f / dt)), g=g, f=f * rotation, grid_step=(dx, dx), dt=dt, outfreq=outfreq
    )
    output = {k: np.asarray(v) for k, v in output.items()}
    output['t'] = np.arange(len(output['h'])) * outfreq * dt / T_f
    return output


def time_mean(output):
    return output['h'][output['t'] > output['t'][-1] - 1].mean(axis=0) - H

### 1. Garb o szerokości $2a$ w kanale (1D): bez rotacji i z rotacją ($a=L_d$, $N=20$)

In [ ]:
N = 20
dx = Ld / N
x = np.arange(-20 * N, 20 * N) * dx
eta_initial = channel(tophat(np.abs(x), a=Ld, w=Ld / 10))
runs = {rotation: simulate(eta_initial=eta_initial, dx=dx, rotation=rotation) for rotation in (False, True)}

for output in runs.values():
    assert abs(output['h'][-1].sum() / output['h'][0].sum() - 1) < 1e-10
    assert np.ptp(output['h'], axis=2).max() < 1e-3 * eta_0

In [ ]:
window = np.abs(x) <= 12 * Ld
levels = np.arange(-.55, 1.06, .1)
fig, axs = pyplot.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for ax, (rotation, output) in zip(axs, runs.items()):
    cnt = ax.contourf(
        x[window] / Ld, output['t'], (output['h'][:, window, 0] - H) / eta_0,
        levels=levels, cmap='RdBu_r', norm=TwoSlopeNorm(vmin=levels[0], vcenter=0, vmax=levels[-1]), extend='both',
    )
    ax.set(xlabel='$x/L_d$', title='z rotacją ($f>0$)' if rotation else 'bez rotacji ($f=0$)')
axs[0].set_ylabel('$t/T_f$')
fig.colorbar(cnt, ax=axs, label=r'$\eta/\eta_0$', ticks=np.arange(-.4, 1.01, .2))
show_plot('hovmoller_1d')

### 2. Stan końcowy a teoria

In [ ]:
eta_num = time_mean(runs[True])
eta_th = adjusted_state(eta_initial, dx)
eta_norot = time_mean(runs[False])

core = np.abs(x) < 5 * Ld
assert np.abs(eta_num - eta_th)[core].max() < .02 * eta_0
assert np.abs(eta_norot[core]).max() < .02 * eta_0

In [ ]:
window = np.abs(x) <= 5 * Ld
pyplot.figure(figsize=(7, 3.5), constrained_layout=True)
pyplot.plot(x[window] / Ld, eta_initial[window, 0] / eta_0, color='gray', linestyle=':', label='stan początkowy')
pyplot.plot(x[window] / Ld, eta_norot[window, 0] / eta_0, label='$f=0$')
pyplot.plot(x[window] / Ld, eta_num[window, 0] / eta_0, label='$f>0$, PyMPDATA')
pyplot.plot(x[window] / Ld, eta_th[window, 0] / eta_0, color='black', linestyle='--', label='$f>0$, teoria')
pyplot.xlabel('$x/L_d$')
pyplot.ylabel(r'$\eta/\eta_0$')
pyplot.legend()
pyplot.grid()
show_plot('profile_1d')

### 3. Ile zostaje w zależności od $a/L_d$

Dla prostokątnego garbu teoria daje w środku $\eta(0)/\eta_0 = 1-e^{-a/L_d}$: garby dużo mniejsze od $L_d$ prawie całkowicie odpływają, dużo większe prawie całkowicie zostają.

In [ ]:
def retained_height(ratio):
    eta_initial = channel(tophat(np.abs(x), a=ratio * Ld, w=Ld / 10))
    centre = len(x) // 2
    return (
        time_mean(simulate(eta_initial=eta_initial, dx=dx))[centre, 0] / eta_0,
        adjusted_state(eta_initial, dx)[centre, 0] / eta_0,
    )


ratios = np.array([.25, .5, 1, 2, 4])
retained, retained_th = np.array([retained_height(ratio) for ratio in ratios]).T

assert np.allclose(retained, retained_th, atol=.02)

In [ ]:
ratios_th = np.linspace(0, 6, 61)
pyplot.figure(figsize=(6, 4), constrained_layout=True)
pyplot.plot(ratios_th, 1 - np.exp(-ratios_th), color='black', linestyle='--', label='teoria: $1-e^{-a/L_d}$')
pyplot.plot(ratios, retained, 'o', label='PyMPDATA')
pyplot.xlabel('$a/L_d$')
pyplot.ylabel(r'$\eta(0)/\eta_0$ po przystosowaniu')
pyplot.ylim(0, 1.05)
pyplot.legend()
pyplot.grid()
show_plot('retained')

### 4. Zbieżność

Stan końcowy dla rozdzielczości $N=L_d/\Delta x = 2\ldots16$ i trzech liczb Couranta fal $c\,\Delta t/\Delta x$ porównany z rozwiązaniem wzorcowym ($N=32$). Linia pozioma: różnica między rozwiązaniem wzorcowym a teorią liniową (efekty nieliniowe i resztkowe oscylacje inercyjne).

In [ ]:
resolutions, courants, reference = np.array([2, 4, 8, 16]), (.1, .2, .4), 32


def final_state(n, courant):
    dx = Ld / n
    xn = np.arange(-20 * n, 20 * n) * dx
    eta_initial = channel(tophat(np.abs(xn), a=Ld, w=Ld / 10))
    return xn, time_mean(simulate(eta_initial=eta_initial, dx=dx, courant=courant))[:, 0], adjusted_state(eta_initial, dx)[:, 0]


def rms(diff, xn):
    return np.sqrt(np.mean(diff[np.abs(xn) < 5 * Ld] ** 2)) / eta_0


x_ref, eta_ref, eta_ref_th = final_state(reference, courants[0])
model_error = rms(eta_ref - eta_ref_th, x_ref)
errors = {}
for courant in courants:
    errors[courant] = []
    for n in resolutions:
        xn, eta, _ = final_state(n, courant)
        errors[courant].append(rms(eta - eta_ref[::reference // n], xn))
    assert np.all(np.diff(np.log2(errors[courant])) < -1)
assert model_error < .01

In [ ]:
pyplot.figure(figsize=(6, 4), constrained_layout=True)
for courant, err in errors.items():
    pyplot.loglog(resolutions, err, 'o-', label=rf'$c\,\Delta t/\Delta x={courant}$')
pyplot.loglog(resolutions, errors[courants[0]][0] * (resolutions[0] / resolutions) ** 2, color='gray', linestyle=':', label=r'$\propto \Delta x^2$')
pyplot.axhline(model_error, color='black', linestyle='--', label='wzorzec – teoria liniowa')
pyplot.xlabel(r'$N = L_d/\Delta x$')
pyplot.ylabel(r'błąd RMS $/\eta_0$')
pyplot.xticks(resolutions, resolutions)
pyplot.tick_params(axis='x', which='minor', labelbottom=False)
pyplot.legend(loc='upper right')
pyplot.grid(which='both')
show_plot('convergence')

### 5. Okrągły garb w 2D ($a = 1.5\,L_d$, $N=12$): animacja

In [ ]:
dx_2d, n = Ld / 12, 240
xy = (np.arange(n) - n // 2) * dx_2d
eta_initial_2d = tophat(np.hypot(*np.meshgrid(xy, xy, indexing='ij')), a=1.5 * Ld, w=2 * dx_2d)
runs_2d = {
    rotation: simulate(eta_initial=eta_initial_2d, dx=dx_2d, rotation=rotation, n_periods=2, outfreq=6)
    for rotation in (False, True)
}
eta_2d_num = time_mean(runs_2d[True])
eta_2d_th = adjusted_state(eta_initial_2d, dx_2d)

core_2d = np.abs(xy) < 4 * Ld
assert abs(eta_2d_num[n // 2, n // 2] - eta_2d_th[n // 2, n // 2]) < .05 * eta_0
assert np.abs(eta_2d_num - eta_2d_th)[core_2d][:, core_2d].max() < .05 * eta_0

In [ ]:
zoom = np.abs(xy) <= 6 * Ld
arrows = slice(3, None, 7)
levels_2d = np.arange(-.35, 1.06, .1)
velocity_2d = [runs_2d[True][k] / runs_2d[True]['h'] for k in ('uh', 'vh')]
arrow_scale = np.hypot(*velocity_2d).max() / .55


def plot(frame):
    fig, axs = pyplot.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
    for ax, (rotation, output) in zip(axs, runs_2d.items()):
        cnt = ax.contourf(
            xy[zoom] / Ld, xy[zoom] / Ld, (output['h'][frame] - H)[zoom][:, zoom].T / eta_0,
            levels=levels_2d, cmap='RdBu_r', norm=TwoSlopeNorm(vmin=levels_2d[0], vcenter=0, vmax=levels_2d[-1]), extend='both',
        )
        ax.set(xlabel='$x/L_d$', ylabel='$y/L_d$', aspect='equal', title=('$f>0$' if rotation else '$f=0$') + rf', $t={output["t"][frame]:.2f}\,T_f$')
    u, v = (component[frame][zoom][:, zoom][arrows, arrows] for component in velocity_2d)
    grid = xy[zoom][arrows] / Ld
    axs[1].quiver(*np.meshgrid(grid, grid, indexing='ij'), u, v, angles='xy', scale_units='xy', scale=arrow_scale, width=.003)
    fig.colorbar(cnt, ax=axs, label=r'$\eta/\eta_0$', ticks=np.arange(-.2, 1.01, .2), shrink=.8)
    return fig


show_anim(plot, range(0, len(runs_2d[True]['t']), 2))